# 4 · Vision SFT
The same as SFT, but **each example includes a picture**.

## When is vision fine-tuning worth it?
| Prompting is enough | Vision fine-tuning helps |
|---|---|
| common objects, plain descriptions | house-specific visual codes, labels or layouts |
| a large vision model is available | a small, local model must do the job |
| | many labeled pictures exist |

## The task: shop tags
The shop marks products with a colored tag. A house rule says what each shape means:
**circle = shirt, square = shoes, triangle = jacket.** The model must write the same command format as notebook 1:

(a red circle tag) → `FIND shirt color=RD`

- **Option A:** prompting only (the rule is written in a long prompt).
- **Option B:** fine-tuning on labeled pictures, then a short prompt.

## Key words
| Word | Meaning | Example |
|---|---|---|
| **image tokens** | the picture is cut into small squares, and each one becomes a token | a small picture → a few dozen tokens |
| **processor** | prepares the input: text → tokens, picture → squares | resizes the picture, then cuts it |
| **epoch** | one pass through all training pictures | 240 pictures × 3 epochs |

## Step 1 · Load the model
The same small model as notebook 1. It can read pictures too.

In [ ]:
import random, warnings, torch
from transformers import AutoModelForImageTextToText, AutoProcessor, logging

warnings.filterwarnings('ignore')             # keep the output readable
logging.set_verbosity_error()

MODEL = 'Qwen/Qwen3.5-0.8B'
proc = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cuda')

## Step 2 · Create the tag pictures
Drawn by code, so the right command is always known. Random size and position make every picture different.
12 kinds of tags (3 shapes × 4 colors): 240 pictures to learn from, 36 new ones to test.

In [ ]:
from PIL import Image, ImageDraw

SHAPES = {'circle': 'shirt', 'square': 'shoes', 'triangle': 'jacket'}   # house rule: shape -> item
COLORS = {'red': 'RD', 'blue': 'BL', 'black': 'BK', 'white': 'WH'}

def draw(shape, color):
    img = Image.new('RGB', (128, 128), 'lightgray')
    x, y, s = random.randint(5, 40), random.randint(5, 40), random.randint(40, 80)   # random place and size
    pen = ImageDraw.Draw(img)
    if shape == 'circle':
        pen.ellipse([x, y, x + s, y + s], fill=color)
    elif shape == 'square':
        pen.rectangle([x, y, x + s, y + s], fill=color)
    else:
        pen.polygon([(x + s / 2, y), (x, y + s), (x + s, y + s)], fill=color)
    return img, f'FIND {SHAPES[shape]} color={COLORS[color]}'

random.seed(0)
KINDS = [(s, c) for s in SHAPES for c in COLORS]   # 12 kinds of tags
TRAIN = [draw(s, c) for s, c in KINDS * 20]        # 240 pictures
TEST = [draw(s, c) for s, c in KINDS * 3]          # 36 new pictures

for img, command in [TRAIN[0], TRAIN[5], TRAIN[10]]:
    display(img)
    print(command)

## Step 3 · A picture becomes tokens
The text holds one placeholder (`<|image_pad|>`). The processor replaces it with the image tokens.

In [ ]:
LONG = '''Read the shop tag in the picture and write a search command.
The shape gives the item: circle = shirt, square = shoes, triangle = jacket.
The color gives the code: red=RD, blue=BL, black=BK, white=WH.
Format: FIND <item> color=<code>
Example: a blue square -> FIND shoes color=BL
Reply with only the command.'''
SHORT = 'Search command for this tag.'

def chat_text(text):
    chat = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': text}]}]
    return proc.apply_chat_template(chat, add_generation_prompt=True)

print(chat_text(SHORT))
x = proc(text=chat_text(SHORT), images=[TRAIN[0][0]], return_tensors='pt')
print('image tokens:', (x['input_ids'] == model.config.image_token_id).sum().item())

## Step 4 · Option A: prompting only
The house rule is written in the prompt. No training.

**Look for:** which part goes wrong: the shape rule, the color code, or the format.

In [ ]:
def ask(model, img, text):
    x = proc(text=chat_text(text), images=[img], return_tensors='pt').to('cuda', torch.bfloat16)
    y = model.generate(**x, max_new_tokens=20, do_sample=False)
    return proc.decode(y[0][x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def evaluate(model, text, show=3):
    model.eval()
    mistakes = []
    for img, command in TEST:
        out = ask(model, img, text)
        if out != command:
            mistakes.append((command, out))
    for command, out in mistakes[:show]:
        print(f'  expected: {command:24} got: {out!r}')
    accuracy = 1 - len(mistakes) / len(TEST)
    print(f'accuracy: {accuracy:.0%}  ({len(TEST) - len(mistakes)} of {len(TEST)} correct)')
    return accuracy

prompt_only = evaluate(model, LONG)

## Step 5 · Option B: prepare the fine-tuning
The same as SFT, plus an `image` column. Only the short prompt is used.
`max_length=None` means the text is never cut, because cutting would break the image tokens.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

data = Dataset.from_list([
    {'image': img,
     'prompt': [{'role': 'user', 'content': SHORT}],
     'completion': [{'role': 'assistant', 'content': command}]} for img, command in TRAIN])

trainer = SFTTrainer(
    model=model,
    processing_class=proc,
    train_dataset=data,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules='all-linear'),   # the small add-on
    args=SFTConfig(max_length=None, num_train_epochs=1, learning_rate=2e-4, logging_steps=10),   # 1 epoch per call
)
trainer.model.print_trainable_parameters()

## Step 6 · Train, one epoch at a time
240 pictures ÷ 8 = 30 steps per epoch. After each epoch, the 36 test pictures are checked.

**Look for:** the accuracy rising epoch by epoch.

In [ ]:
print('before training (short prompt):')
fine_tuned = evaluate(trainer.model, SHORT)

for epoch in [1, 2, 3]:
    trainer.train()                                 # one pass over the 240 pictures
    print(f'\nafter epoch {epoch}:')
    fine_tuned = evaluate(trainer.model, SHORT)

## Step 7 · Did it work?
It worked if the `loss` went down during training **and** the fine-tuned model beats prompting only on the 36 test pictures.

In [ ]:
print(f'A. prompting only: {prompt_only:.0%} correct')
print(f'B. fine-tuned:     {fine_tuned:.0%} correct')

if prompt_only >= 0.95:
    print('\nPrompting alone was already enough: no training needed.')
elif fine_tuned - prompt_only >= 0.10:
    print('\nIt worked: training clearly beat prompting only.')
else:
    print('\nNo clear gain: try more epochs or more pictures.')

## Summary
- **Needs:** labeled pictures (here 240)
- **Learns:** a visual house rule, from examples
- **Progress:** the loss goes down, the test accuracy goes up
- **Keep in mind:** the same idea works for real cases, such as reading a company's forms, labels or screenshots